# Coil Centering Scan Notebook

This notebook scans how well the bias-corrected magnetic-axis estimate recovers the actual transformed field-coil geometry as a function of:

- Hall probe count/layout
- Coil current
- Applied shifts and tilts
- Optional magnetic field noise

It uses the same core definitions as `coil_centering_full.jl` and `tilt_shift.jl`:

- geometric center = segment-length-weighted centroid
- geometric orientation = weighted best-fit plane normal
- magnetic `z`/tilt = `B_R` zero-crossing sinusoid
- magnetic `x/y` = local `B_phi` minimization
- bias correction = forward-model bias subtraction using assumed physical coil model


In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))

using GeneralizedPerturbedEquilibrium
using LinearAlgebra
using Statistics
using Printf
using Random

for pkg in ["DataFrames", "CSV", "CairoMakie"]
    try
        @eval using $(Symbol(pkg))
    catch
        Pkg.add(pkg)
        @eval using $(Symbol(pkg))
    end
end

const FT = GeneralizedPerturbedEquilibrium.ForcingTerms
const compute_biot_savart_boundary! = FT.compute_biot_savart_boundary!
const read_coil_dat = FT.read_coil_dat

println("Notebook setup complete.")

  Activating project at `~/Documents/GitHub/JULIA_GPEC`
    Updating registry at `~/.julia/registries/FuseRegistry`
    Updating git-repo `https://github.com/ProjectTorreyPines/FuseRegistry.jl.git`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Graphics ─── v1.1.3
   Installed CairoMakie ─ v0.15.9
   Installed Cairo ────── v1.1.1
      Compat entries added for CairoMakie
    Updating `~/Documents/GitHub/JULIA_GPEC/Project.toml`
⌃ [13f3f980] + CairoMakie v0.15.9
    Updating `~/Documents/GitHub/JULIA_GPEC/Manifest.toml`
  [159f3aea] + Cairo v1.1.1
⌃ [13f3f980] + CairoMakie v0.15.9
  [a2bd30eb] + Graphics v1.1.3
        Info Packages marked with ⌃ have new versions available and may be upgradable.


Notebook setup complete.


Precompiling packages...
   1007.6 ms  ✓ Graphics
   4723.0 ms  ✓ Cairo
  38673.9 ms  ✓ CairoMakie
  3 dependencies successfully precompiled in 47 seconds. 628 already precompiled.


## User configuration

Edit these paths and scan settings first.

In [ ]:
# Paths
BASE_DIR = @__DIR__
PHYSICAL_COIL_FILE = joinpath(BASE_DIR, "sparc_pf1u.dat")
SCAN_OUTPUT_DIR = joinpath(BASE_DIR, "coil_centering_scan_outputs")
mkpath(SCAN_OUTPUT_DIR)

# Base transform used by tilt_shift.jl. Scans below multiply these by scale factors.
BASE_SHIFT_X = 0.01   # m
BASE_SHIFT_Y = 0.03   # m
BASE_SHIFT_Z = 0.05   # m
BASE_TILT_X_DEG = 0.1
BASE_TILT_Y_DEG = 0.5
BASE_TILT_Z_DEG = 0.0

# Hall-probe shell radii and z-spans
HALL_R_INNER_FRAC = 0.40
HALL_R_OUTER_FRAC = 0.50
HALL_Z_HALFSPAN_INNER_M = 0.60
HALL_Z_HALFSPAN_OUTER_M = 0.40

# Analysis settings
TILT_CALIBRATION_RADIUS_M = NaN
BR_ZERO_NOISE_FRAC = 0.0
XY_MIN_STEP0_M = 0.050
XY_MIN_TOL_M = 1e-8
BIAS_CORRECTION_N_ITER = 4
BIAS_CORRECTION_DAMPING = 0.6

# Optional synthetic measurement noise added to each field component.
# If these are both zero, current will mostly not matter because Biot-Savart is linear in current.
B_NOISE_ABS_T = 0.0       # absolute Tesla noise
B_NOISE_REL = 0.0         # relative to median |B|
RNG_SEED = 1234

# Scan grids. Keep these small first; each row does multiple Biot-Savart solves.
#hall_layouts = [
#    (name="coarse",  nphi_i=12, nz_i=10, nphi_o=10, nz_o=6),
#    (name="medium",  nphi_i=24, nz_i=22, nphi_o=20, nz_o=10),
#    (name="dense",   nphi_i=36, nz_i=30, nphi_o=30, nz_o=16),
#    ]
hall_layouts = [
    (name="coarse",  nphi_i=6, nz_i=5, nphi_o=5, nz_o=3),
    (name="medium",  nphi_i=12, nz_i=10, nphi_o=10, nz_o=6),
    (name="dense",   nphi_i=24, nz_i=22, nphi_o=20, nz_o=10),
]

coil_currents_A = [10.0, 100.0, 1000.0]
transform_scales = [0.0, 0.25, 0.50, 1.0]

println("Physical coil: ", PHYSICAL_COIL_FILE)
println("Scan output dir: ", SCAN_OUTPUT_DIR)

Physical coil: /Users/bursche/Documents/GitHub/JULIA_GPEC/examples/Br_3D_example/sparc_pf1u.dat
Scan output dir: /Users/bursche/Documents/GitHub/JULIA_GPEC/examples/Br_3D_example/coil_centering_scan_outputs


## Helper functions

This cell contains the transform utility, geometric pose calculation, Hall field generation, magnetic-axis analysis, and bias correction.

In [13]:
wmean(x, w) = sum(w) <= 0 ? mean(x) : sum(w .* x) / sum(w)

function meanfinite(v)
    u = filter(isfinite, collect(v))
    isempty(u) ? NaN : mean(u)
end

function zcrossings(vals, z)
    out = Float64[]
    for i in 1:(length(vals)-1)
        vals[i] == 0 && push!(out, z[i])
        vals[i] * vals[i+1] < 0 || continue
        t = vals[i] / (vals[i] - vals[i+1])
        push!(out, z[i] + t * (z[i+1] - z[i]))
    end
    out
end

function rotation_x(theta)
    c, s = cos(theta), sin(theta)
    [1.0 0.0 0.0; 0.0 c -s; 0.0 s c]
end

function rotation_y(theta)
    c, s = cos(theta), sin(theta)
    [c 0.0 s; 0.0 1.0 0.0; -s 0.0 c]
end

function rotation_z(theta)
    c, s = cos(theta), sin(theta)
    [c -s 0.0; s c 0.0; 0.0 0.0 1.0]
end

build_rotation_matrix(tx_deg, ty_deg, tz_deg) = rotation_z(deg2rad(tz_deg)) * rotation_y(deg2rad(ty_deg)) * rotation_x(deg2rad(tx_deg))

function read_raw_coil_file(filepath::String)
    isfile(filepath) || error("Input coil file not found: $filepath")
    lines = readlines(filepath)
    isempty(lines) && error("Coil file is empty: $filepath")
    header = lines[1]
    x = Float64[]; y = Float64[]; z = Float64[]
    for i in 2:length(lines)
        s = strip(lines[i])
        isempty(s) && continue
        tok = split(s)
        length(tok) >= 3 || continue
        try
            push!(x, parse(Float64, tok[1]))
            push!(y, parse(Float64, tok[2]))
            push!(z, parse(Float64, tok[3]))
        catch
        end
    end
    return header, x, y, z
end

function write_raw_coil_file(filepath::String, header::String, x, y, z)
    open(filepath, "w") do io
        println(io, header)
        for i in eachindex(x)
            @printf(io, "  %14.7e   %14.7e   %14.7e\n", x[i], y[i], z[i])
        end
    end
end

function is_closed_curve(x, y, z; tol=1e-12)
    length(x) >= 2 || return false
    return (x[1]-x[end])^2 + (y[1]-y[end])^2 + (z[1]-z[end])^2 <= tol^2
end

function raw_segment_midpoints_lengths(x, y, z)
    N = length(x)
    N >= 2 || error("Need at least two coil points")
    closed = is_closed_curve(x, y, z)
    lastseg = closed ? N-1 : N
    mx = Float64[]; my = Float64[]; mz = Float64[]; w = Float64[]
    for i in 1:lastseg
        j = i == N ? 1 : i+1
        dlx = x[j]-x[i]; dly = y[j]-y[i]; dlz = z[j]-z[i]
        ds = sqrt(dlx^2+dly^2+dlz^2)
        ds <= 0 && continue
        push!(mx, 0.5*(x[i]+x[j]))
        push!(my, 0.5*(y[i]+y[j]))
        push!(mz, 0.5*(z[i]+z[j]))
        push!(w, ds)
    end
    return mx, my, mz, w
end

function plane_basis_from_normal(n)
    ref = abs(n[3]) < 0.9 ? [0.0,0.0,1.0] : [1.0,0.0,0.0]
    e1 = normalize(cross(ref, n))
    e2 = cross(n, e1)
    return e1, e2
end

function raw_pose_from_points(x, y, z)
    mx, my, mz, w = raw_segment_midpoints_lengths(x, y, z)
    W = sum(w)
    cx = sum(w .* mx) / W
    cy = sum(w .* my) / W
    cz = sum(w .* mz) / W
    P = hcat(mx .- cx, my .- cy, mz .- cz)
    Pw = P .* sqrt.(w)
    _, _, V = svd(Pw)
    n = Vector(V[:,3])
    n[3] < 0 && (n = -n)
    tx = atand(-n[2], n[3])
    ty = atand( n[1], n[3])
    return (x0=cx, y0=cy, z0=cz, tilt_x=tx, tilt_y=ty, normal=n)
end

function transform_coil_file(input_file, output_file; shift_x=0.0, shift_y=0.0, shift_z=0.0, tilt_x_deg=0.0, tilt_y_deg=0.0, tilt_z_deg=0.0)
    header, x, y, z = read_raw_coil_file(input_file)
    pose = raw_pose_from_points(x, y, z)
    R = build_rotation_matrix(tilt_x_deg, tilt_y_deg, tilt_z_deg)
    xnew = similar(x); ynew = similar(y); znew = similar(z)
    c = [pose.x0, pose.y0, pose.z0]
    for i in eachindex(x)
        q = R * ([x[i], y[i], z[i]] - c) + c + [shift_x, shift_y, shift_z]
        xnew[i] = q[1]; ynew[i] = q[2]; znew[i] = q[3]
    end
    write_raw_coil_file(output_file, header, xnew, ynew, znew)
    return output_file
end

function load_coils(file; current_A=100.0)
    cs = read_coil_dat(file)
    cs.currents .= current_A
    return [cs]
end

function segment_midpoints_lengths(coils)
    mx = Float64[]; my = Float64[]; mz = Float64[]; w = Float64[]
    for cs in coils, j in 1:cs.ncoil, k in 1:cs.s, l in 1:(cs.nsec-1)
        x1 = cs.x[j,k,l];   y1 = cs.y[j,k,l];   z1 = cs.z[j,k,l]
        x2 = cs.x[j,k,l+1]; y2 = cs.y[j,k,l+1]; z2 = cs.z[j,k,l+1]
        ds = sqrt((x2-x1)^2 + (y2-y1)^2 + (z2-z1)^2)
        ds <= 0 && continue
        push!(mx, 0.5*(x1+x2)); push!(my, 0.5*(y1+y2)); push!(mz, 0.5*(z1+z2)); push!(w, ds)
    end
    return mx, my, mz, w
end

function collect_coil_xyz(coils)
    x = Float64[]; y = Float64[]; z = Float64[]
    for cs in coils, j in 1:cs.ncoil, k in 1:cs.s
        append!(x, vec(cs.x[j,k,:])); append!(y, vec(cs.y[j,k,:])); append!(z, vec(cs.z[j,k,:]))
    end
    return x, y, z
end

function axis_from_coil(coils)
    mx, my, mz, w = segment_midpoints_lengths(coils)
    W = sum(w)
    cx = sum(w .* mx) / W
    cy = sum(w .* my) / W
    cz = sum(w .* mz) / W
    P = hcat(mx .- cx, my .- cy, mz .- cz)
    Pw = P .* sqrt.(w)
    _, _, V = svd(Pw)
    n = Vector(V[:,3])
    n[3] < 0 && (n = -n)
    tx = atand(-n[2], n[3])
    ty = atand( n[1], n[3])
    e1, e2 = plane_basis_from_normal(n)
    dx = mx .- cx; dy = my .- cy; dz = mz .- cz
    u = dx .* e1[1] .+ dy .* e1[2] .+ dz .* e1[3]
    v = dx .* e2[1] .+ dy .* e2[2] .+ dz .* e2[3]
    rho = sqrt.(u.^2 .+ v.^2)
    rfit = sum(w .* rho) / W
    x, y, z = collect_coil_xyz(coils)
    R = sqrt.(x.^2 .+ y.^2)
    return (x0=cx, y0=cy, z0=cz, tilt_x=tx, tilt_y=ty, tilt_mag=hypot(tx,ty), Rfit=rfit, Rmin=minimum(R), Rmax=maximum(R), Zmin=minimum(z), Zmax=maximum(z))
end

function rotation_between_vectors(a::Vector{Float64}, b::Vector{Float64})
    a = normalize(a); b = normalize(b)
    c = clamp(dot(a,b), -1.0, 1.0)
    abs(c - 1.0) < 1e-12 && return Matrix{Float64}(I,3,3)
    if abs(c + 1.0) < 1e-10
        perp = abs(a[1]) < 0.9 ? [1.0,0.0,0.0] : [0.0,1.0,0.0]
        ax = normalize(cross(a, perp))
        return 2.0 * (ax * ax') - Matrix{Float64}(I,3,3)
    end
    k = normalize(cross(a,b))
    s = sqrt(max(1.0-c^2, 0.0))
    K = [0.0 -k[3] k[2]; k[3] 0.0 -k[1]; -k[2] k[1] 0.0]
    return Matrix{Float64}(I,3,3) + s*K + (1.0-c)*K*K
end

function place_coil_at_axis(coils, x0, y0, z0, tx_deg, ty_deg)
    new = deepcopy(coils)
    p = axis_from_coil(new)
    n_orig = normalize([tand(p.tilt_y), -tand(p.tilt_x), 1.0])
    n_target = normalize([tand(ty_deg), -tand(tx_deg), 1.0])
    zhat = [0.0,0.0,1.0]
    Rtot = rotation_between_vectors(zhat, n_target) * rotation_between_vectors(n_orig, zhat)
    for cs in new, j in 1:cs.ncoil, k in 1:cs.s, l in 1:cs.nsec
        q = Rtot * [cs.x[j,k,l]-p.x0, cs.y[j,k,l]-p.y0, cs.z[j,k,l]-p.z0]
        cs.x[j,k,l] = q[1] + x0; cs.y[j,k,l] = q[2] + y0; cs.z[j,k,l] = q[3] + z0
    end
    return new
end

function make_hall_data(source_coils, phys_ref, layout; current_A=100.0, noise_abs_T=0.0, noise_rel=0.0, seed=1)
    Rinner = HALL_R_INNER_FRAC * phys_ref.Rmin
    Router = HALL_R_OUTER_FRAC * phys_ref.Rmax
    Zc = phys_ref.z0
    phii = collect(range(0, 2pi, length=layout.nphi_i+1)[1:end-1])
    phio = collect(range(0, 2pi, length=layout.nphi_o+1)[1:end-1])
    Zi = collect(range(Zc-HALL_Z_HALFSPAN_INNER_M, Zc+HALL_Z_HALFSPAN_INNER_M; length=layout.nz_i))
    Zo = collect(range(Zc-HALL_Z_HALFSPAN_OUTER_M, Zc+HALL_Z_HALFSPAN_OUTER_M; length=layout.nz_o))
    N = layout.nphi_i*layout.nz_i + layout.nphi_o*layout.nz_o
    R = zeros(N); phi = zeros(N); Z = zeros(N); shell = zeros(Int,N)
    idx = 1
    for iz in eachindex(Zi), ip in eachindex(phii)
        R[idx]=Rinner; phi[idx]=phii[ip]; Z[idx]=Zi[iz]; shell[idx]=1; idx+=1
    end
    for iz in eachindex(Zo), ip in eachindex(phio)
        R[idx]=Router; phi[idx]=phio[ip]; Z[idx]=Zo[iz]; shell[idx]=2; idx+=1
    end
    BR = zeros(N); BP = zeros(N); BZ = zeros(N)
    compute_biot_savart_boundary!(BR, BP, BZ, R, phi, Z, source_coils)
    Bmag = sqrt.(BR.^2 .+ BP.^2 .+ BZ.^2)
    sig = noise_abs_T + noise_rel * median(Bmag)
    if sig > 0
        rng = MersenneTwister(seed)
        BR .+= sig .* randn(rng, N)
        BP .+= sig .* randn(rng, N)
        BZ .+= sig .* randn(rng, N)
        Bmag = sqrt.(BR.^2 .+ BP.^2 .+ BZ.^2)
    end
    return (R=R, phi=phi, Z=Z, x=R.*cos.(phi), y=R.*sin.(phi), shell=shell, B_R=BR, B_phi=BP, B_Z=BZ, B_mag=Bmag, Zi=Zi, Zo=Zo, phi_i=phii, phi_o=phio, nzi=layout.nz_i, nzo=layout.nz_o, R_inner=Rinner, R_outer=Router, layout=layout)
end

function shellmeta(h, shell)
    shell == 1 ? (h.layout.nphi_i, h.nzi, h.R_inner, h.Zi, h.phi_i) : (h.layout.nphi_o, h.nzo, h.R_outer, h.Zo, h.phi_o)
end

function magnetic_z_tilt(h, phys_ref; shell=1)
    nphi, nz, Rsh, Zgrid, phigrid = shellmeta(h, shell)
    BR = reshape(h.B_R[h.shell .== shell], nphi, nz)

    zc = fill(NaN, nphi)
    valid = falses(nphi)

    noise_cut = BR_ZERO_NOISE_FRAC * maximum(abs.(BR))

    for ip in 1:nphi
        maximum(abs.(BR[ip, :])) < noise_cut && continue

        c = zcrossings(BR[ip, :], Zgrid)
        isempty(c) && continue

        zc[ip] = c[argmin(abs.(c .- phys_ref.z0))]
        valid[ip] = true
    end

    Rcal = isfinite(TILT_CALIBRATION_RADIUS_M) ? TILT_CALIBRATION_RADIUS_M : phys_ref.Rfit
    nvalid = Base.count(valid)

    if nvalid >= 4
        M = hcat(
            ones(nvalid),
            cos.(phigrid[valid]),
            sin.(phigrid[valid]),
        )

        Z0, Bc, Cs = M \ zc[valid]
        fit = M * [Z0, Bc, Cs]

        tx = atand(Cs, Rcal)
        ty = atand(-Bc, Rcal)
        rms = sqrt(mean((zc[valid] .- fit).^2))

        return (
            shell=shell,
            R=Rsh,
            x0=NaN,
            y0=NaN,
            z0=Z0,
            tilt_x=tx,
            tilt_y=ty,
            tilt_mag=hypot(tx, ty),
            rms=rms,
        )
    else
        return (
            shell=shell,
            R=Rsh,
            x0=NaN,
            y0=NaN,
            z0=NaN,
            tilt_x=NaN,
            tilt_y=NaN,
            tilt_mag=NaN,
            rms=NaN,
        )
    end
end

function B_cartesian(h)
    c = cos.(h.phi); s = sin.(h.phi)
    Bx = h.B_R .* c .- h.B_phi .* s
    By = h.B_R .* s .+ h.B_phi .* c
    return Bx, By, h.B_Z
end

function local_bphi_objective(h, idxs, x0, y0, z0, tx_deg, ty_deg)
    Bx, By, Bz = B_cartesian(h)
    d = normalize([tand(ty_deg), -tand(tx_deg), 1.0])
    acc = 0.0; n = 0
    for i in idxs
        qx = h.x[i]-x0; qy = h.y[i]-y0; qz = h.Z[i]-z0
        s = qx*d[1] + qy*d[2] + qz*d[3]
        rx = qx - s*d[1]; ry = qy - s*d[2]; rz = qz - s*d[3]
        rho = sqrt(rx^2+ry^2+rz^2)
        rho < 1e-9 && continue
        rhat = [rx/rho, ry/rho, rz/rho]
        ephi = cross(d, rhat)
        bp = Bx[i]*ephi[1] + By[i]*ephi[2] + Bz[i]*ephi[3]
        acc += (bp / max(h.B_mag[i], 1e-30))^2
        n += 1
    end
    acc / max(n,1)
end

function magnetic_xy_minimize(h, phys_ref, zt; shell=1)
    idxs = findall(h.shell .== shell)
    z0 = isfinite(zt.z0) ? zt.z0 : phys_ref.z0
    tx = isfinite(zt.tilt_x) ? zt.tilt_x : phys_ref.tilt_x
    ty = isfinite(zt.tilt_y) ? zt.tilt_y : phys_ref.tilt_y
    x = phys_ref.x0; y = phys_ref.y0
    step = XY_MIN_STEP0_M
    best = local_bphi_objective(h, idxs, x, y, z0, tx, ty)
    dirs = [(1.0,0.0),(-1.0,0.0),(0.0,1.0),(0.0,-1.0),(1.0,1.0),(1.0,-1.0),(-1.0,1.0),(-1.0,-1.0)]
    while step > XY_MIN_TOL_M
        improved = false
        for (dx,dy) in dirs
            xn = x + step*dx; yn = y + step*dy
            val = local_bphi_objective(h, idxs, xn, yn, z0, tx, ty)
            if val < best
                x=xn; y=yn; best=val; improved=true
            end
        end
        improved || (step *= 0.5)
    end
    return (shell=shell, R=zt.R, x0=x, y0=y, z0=z0, tilt_x=tx, tilt_y=ty, tilt_mag=hypot(tx,ty), objective=best)
end

function average_axes(a1, a2)
    tx = meanfinite([a1.tilt_x, a2.tilt_x]); ty = meanfinite([a1.tilt_y, a2.tilt_y])
    return (shell=0, R=NaN, x0=meanfinite([a1.x0,a2.x0]), y0=meanfinite([a1.y0,a2.y0]), z0=meanfinite([a1.z0,a2.z0]), tilt_x=tx, tilt_y=ty, tilt_mag=hypot(tx,ty))
end

function run_analysis_pipeline(h, phys_ref)
    zt1 = magnetic_z_tilt(h, phys_ref; shell=1)
    zt2 = magnetic_z_tilt(h, phys_ref; shell=2)
    sin1 = magnetic_xy_minimize(h, phys_ref, zt1; shell=1)
    sin2 = magnetic_xy_minimize(h, phys_ref, zt2; shell=2)
    return (zt1=zt1, zt2=zt2, sin1=sin1, sin2=sin2, sinavg=average_axes(sin1,sin2))
end

function recompute_hall_field(h, coils)
    BR = zeros(length(h.R)); BP = zeros(length(h.R)); BZ = zeros(length(h.R))
    compute_biot_savart_boundary!(BR, BP, BZ, h.R, h.phi, h.Z, coils)
    Bmag = sqrt.(BR.^2 .+ BP.^2 .+ BZ.^2)
    return merge(h, (B_R=BR, B_phi=BP, B_Z=BZ, B_mag=Bmag))
end

function bias_correct_axis(model_coils, h, measured_zt1, measured_zt2, measured_sin1, measured_sin2)
    meas_x  = meanfinite([measured_sin1.x0, measured_sin2.x0])
    meas_y  = meanfinite([measured_sin1.y0, measured_sin2.y0])
    meas_z  = meanfinite([measured_zt1.z0, measured_zt2.z0])
    meas_tx = meanfinite([measured_zt1.tilt_x, measured_zt2.tilt_x])
    meas_ty = meanfinite([measured_zt1.tilt_y, measured_zt2.tilt_y])
    any(x -> !isfinite(x), [meas_x,meas_y,meas_z,meas_tx,meas_ty]) && return average_axes(measured_sin1, measured_sin2)
    est_x, est_y, est_z, est_tx, est_ty = meas_x, meas_y, meas_z, meas_tx, meas_ty
    corrected = (x0=meas_x, y0=meas_y, z0=meas_z, tilt_x=meas_tx, tilt_y=meas_ty, tilt_mag=hypot(meas_tx,meas_ty))
    for iter in 1:BIAS_CORRECTION_N_ITER
        synth = place_coil_at_axis(model_coils, est_x, est_y, est_z, est_tx, est_ty)
        hs = recompute_hall_field(h, synth)
        gs = axis_from_coil(synth)
        rs = run_analysis_pipeline(hs, gs)
        bx  = meanfinite([rs.sin1.x0, rs.sin2.x0]) - gs.x0
        by  = meanfinite([rs.sin1.y0, rs.sin2.y0]) - gs.y0
        bz  = meanfinite([rs.zt1.z0, rs.zt2.z0]) - gs.z0
        btx = meanfinite([rs.zt1.tilt_x, rs.zt2.tilt_x]) - gs.tilt_x
        bty = meanfinite([rs.zt1.tilt_y, rs.zt2.tilt_y]) - gs.tilt_y
        cx = meas_x - bx; cy = meas_y - by; cz = meas_z - bz; ctx = meas_tx - btx; cty = meas_ty - bty
        corrected = (x0=cx, y0=cy, z0=cz, tilt_x=ctx, tilt_y=cty, tilt_mag=hypot(ctx,cty))
        alpha = BIAS_CORRECTION_DAMPING
        est_x += alpha*(cx-est_x); est_y += alpha*(cy-est_y); est_z += alpha*(cz-est_z); est_tx += alpha*(ctx-est_tx); est_ty += alpha*(cty-est_ty)
    end
    return corrected
end

function analyze_one_case(; scale, layout, current_A, noise_abs_T=B_NOISE_ABS_T, noise_rel=B_NOISE_REL, seed=RNG_SEED)
    tag = @sprintf("scale_%05.2f_%s_I_%g", scale, layout.name, current_A)
    field_file = joinpath(SCAN_OUTPUT_DIR, "field_$(replace(tag, '.' => 'p')).dat")
    sx = scale * BASE_SHIFT_X; sy = scale * BASE_SHIFT_Y; sz = scale * BASE_SHIFT_Z
    tx = scale * BASE_TILT_X_DEG; ty = scale * BASE_TILT_Y_DEG; tz = scale * BASE_TILT_Z_DEG
    transform_coil_file(PHYSICAL_COIL_FILE, field_file; shift_x=sx, shift_y=sy, shift_z=sz, tilt_x_deg=tx, tilt_y_deg=ty, tilt_z_deg=tz)
    physical_coils = load_coils(PHYSICAL_COIL_FILE; current_A=current_A)
    field_coils = load_coils(field_file; current_A=current_A)
    model_coils = load_coils(PHYSICAL_COIL_FILE; current_A=current_A)
    phys_geom = axis_from_coil(physical_coils)
    field_geom = axis_from_coil(field_coils)
    h = make_hall_data(field_coils, phys_geom, layout; current_A=current_A, noise_abs_T=noise_abs_T, noise_rel=noise_rel, seed=seed)
    raw = run_analysis_pipeline(h, phys_geom)
    corr = bias_correct_axis(model_coils, h, raw.zt1, raw.zt2, raw.sin1, raw.sin2)
    return (
        scale=scale, layout=layout.name, nprobes=layout.nphi_i*layout.nz_i + layout.nphi_o*layout.nz_o, current_A=current_A,
        input_shift_x_mm=sx*1e3, input_shift_y_mm=sy*1e3, input_shift_z_mm=sz*1e3,
        input_tilt_x_deg=tx, input_tilt_y_deg=ty, input_tilt_z_deg=tz,
        field_geom_dx_mm=(field_geom.x0-phys_geom.x0)*1e3,
        field_geom_dy_mm=(field_geom.y0-phys_geom.y0)*1e3,
        field_geom_dz_mm=(field_geom.z0-phys_geom.z0)*1e3,
        field_geom_dtx_deg=field_geom.tilt_x-phys_geom.tilt_x,
        field_geom_dty_deg=field_geom.tilt_y-phys_geom.tilt_y,
        raw_err_x_mm=(raw.sinavg.x0-field_geom.x0)*1e3,
        raw_err_y_mm=(raw.sinavg.y0-field_geom.y0)*1e3,
        raw_err_z_mm=(raw.sinavg.z0-field_geom.z0)*1e3,
        raw_err_tilt_x_deg=raw.sinavg.tilt_x-field_geom.tilt_x,
        raw_err_tilt_y_deg=raw.sinavg.tilt_y-field_geom.tilt_y,
        corr_err_x_mm=(corr.x0-field_geom.x0)*1e3,
        corr_err_y_mm=(corr.y0-field_geom.y0)*1e3,
        corr_err_z_mm=(corr.z0-field_geom.z0)*1e3,
        corr_err_tilt_x_deg=corr.tilt_x-field_geom.tilt_x,
        corr_err_tilt_y_deg=corr.tilt_y-field_geom.tilt_y,
        corr_pos_err_mm=sqrt(((corr.x0-field_geom.x0)*1e3)^2 + ((corr.y0-field_geom.y0)*1e3)^2 + ((corr.z0-field_geom.z0)*1e3)^2),
        corr_tilt_err_deg=hypot(corr.tilt_x-field_geom.tilt_x, corr.tilt_y-field_geom.tilt_y),
        raw_pos_err_mm=sqrt(((raw.sinavg.x0-field_geom.x0)*1e3)^2 + ((raw.sinavg.y0-field_geom.y0)*1e3)^2 + ((raw.sinavg.z0-field_geom.z0)*1e3)^2),
        raw_tilt_err_deg=hypot(raw.sinavg.tilt_x-field_geom.tilt_x, raw.sinavg.tilt_y-field_geom.tilt_y)
    )
end

println("Helper functions loaded.")

Helper functions loaded.


## Run the scan

This can take a while. Start small, then increase the scan arrays above.

In [14]:
rows = NamedTuple[]
total = length(transform_scales) * length(hall_layouts) * length(coil_currents_A)
case_idx = 0

for scale in transform_scales
    for layout in hall_layouts
        for Iamp in coil_currents_A
            global case_idx += 1

            @printf(
                "[%d/%d] scale=%.2f layout=%s I=%.1f A\n",
                case_idx, total, scale, layout.name, Iamp,
            )

            try
                push!(
                    rows,
                    analyze_one_case(
                        scale=scale,
                        layout=layout,
                        current_A=Iamp,
                        seed=RNG_SEED + case_idx,
                    ),
                )
            catch err
                @warn "case failed" scale layout=layout.name Iamp exception=(err, catch_backtrace())
            end
        end
    end
end

df = DataFrame(rows)
csv_path = joinpath(SCAN_OUTPUT_DIR, "coil_centering_scan_results.csv")
CSV.write(csv_path, df)

println("Saved: ", csv_path)
df

[1/36] scale=0.00 layout=coarse I=10.0 A
[2/36] scale=0.00 layout=coarse I=100.0 A
[3/36] scale=0.00 layout=coarse I=1000.0 A
[4/36] scale=0.00 layout=medium I=10.0 A
[5/36] scale=0.00 layout=medium I=100.0 A
[6/36] scale=0.00 layout=medium I=1000.0 A
[7/36] scale=0.00 layout=dense I=10.0 A
[8/36] scale=0.00 layout=dense I=100.0 A
[9/36] scale=0.00 layout=dense I=1000.0 A
[10/36] scale=0.25 layout=coarse I=10.0 A
[11/36] scale=0.25 layout=coarse I=100.0 A
[12/36] scale=0.25 layout=coarse I=1000.0 A
[13/36] scale=0.25 layout=medium I=10.0 A
[14/36] scale=0.25 layout=medium I=100.0 A
[15/36] scale=0.25 layout=medium I=1000.0 A
[16/36] scale=0.25 layout=dense I=10.0 A
[17/36] scale=0.25 layout=dense I=100.0 A
[18/36] scale=0.25 layout=dense I=1000.0 A
[19/36] scale=0.50 layout=coarse I=10.0 A
[20/36] scale=0.50 layout=coarse I=100.0 A
[21/36] scale=0.50 layout=coarse I=1000.0 A
[22/36] scale=0.50 layout=medium I=10.0 A
[23/36] scale=0.50 layout=medium I=100.0 A
[24/36] scale=0.50 layout=m

Row,scale,layout,nprobes,current_A,input_shift_x_mm,input_shift_y_mm,input_shift_z_mm,input_tilt_x_deg,input_tilt_y_deg,input_tilt_z_deg,field_geom_dx_mm,field_geom_dy_mm,field_geom_dz_mm,field_geom_dtx_deg,field_geom_dty_deg,raw_err_x_mm,raw_err_y_mm,raw_err_z_mm,raw_err_tilt_x_deg,raw_err_tilt_y_deg,corr_err_x_mm,corr_err_y_mm,corr_err_z_mm,corr_err_tilt_x_deg,corr_err_tilt_y_deg,corr_pos_err_mm,corr_tilt_err_deg,raw_pos_err_mm,raw_tilt_err_deg
,Float64,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,coarse,180,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.35671,-3.87483,-0.145189,-0.0393917,-0.132606,0.000572205,0.000786781,-0.0015186,3.19815e-7,2.30658e-7,0.00180349,3.94315e-7,4.10804,0.138333
2,0.0,coarse,180,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.35671,-3.87483,-0.145189,-0.0393917,-0.132606,0.000572205,0.000786781,-0.0015186,3.19815e-7,2.30658e-7,0.00180349,3.94315e-7,4.10804,0.138333
3,0.0,coarse,180,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.35671,-3.87483,-0.145189,-0.0393917,-0.132606,0.000572205,0.000786781,-0.0015186,3.19815e-7,2.30658e-7,0.00180349,3.94315e-7,4.10804,0.138333
4,0.0,medium,728,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36771,-3.92873,-0.014767,-0.0244262,-0.134916,0.000631809,0.000935793,-0.000768411,-4.95424e-9,-1.50572e-6,0.00136578,1.50573e-6,4.16002,0.137109
5,0.0,medium,728,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36771,-3.92873,-0.014767,-0.0244262,-0.134916,0.000631809,0.000935793,-0.000768411,-4.95426e-9,-1.50572e-6,0.00136578,1.50573e-6,4.16002,0.137109
6,0.0,medium,728,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36771,-3.92873,-0.014767,-0.0244262,-0.134916,0.000631809,0.000935793,-0.000768411,-4.95426e-9,-1.50572e-6,0.00136578,1.50573e-6,4.16002,0.137109
7,0.0,dense,1560,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36304,-3.9552,0.0206076,-0.0205858,-0.135862,0.00064373,0.000959635,-0.000744284,-2.39775e-7,-3.2601e-6,0.0013745,3.26891e-6,4.18353,0.137413
8,0.0,dense,1560,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36304,-3.9552,0.0206076,-0.0205858,-0.135862,0.00064373,0.000959635,-0.000744284,-2.39775e-7,-3.2601e-6,0.0013745,3.26891e-6,4.18353,0.137413
9,0.0,dense,1560,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.36304,-3.9552,0.0206076,-0.0205858,-0.135862,0.00064373,0.000959635,-0.000744284,-2.39775e-7,-3.2601e-6,0.0013745,3.26891e-6,4.18353,0.137413


## Compact diagnostic tables

In [15]:
# Best/worst corrected recovery cases
sort(df, :corr_pos_err_mm)[:, [:scale, :layout, :nprobes, :current_A, :corr_pos_err_mm, :corr_tilt_err_deg, :raw_pos_err_mm, :raw_tilt_err_deg]]

Row,scale,layout,nprobes,current_A,corr_pos_err_mm,corr_tilt_err_deg,raw_pos_err_mm,raw_tilt_err_deg
,Float64,String,Int64,Float64,Float64,Float64,Float64,Float64
1,0.5,dense,1560,1000.0,0.00106826,3.24514e-6,3.94477,0.0525965
2,0.5,dense,1560,100.0,0.00106826,3.24514e-6,3.94477,0.0525965
3,0.5,dense,1560,10.0,0.00106826,3.24514e-6,3.94477,0.0525965
4,0.5,medium,728,100.0,0.00110577,2.71689e-6,3.95331,0.0513307
5,0.5,medium,728,10.0,0.00110577,2.71689e-6,3.95331,0.0513307
6,0.5,medium,728,1000.0,0.00110577,2.71689e-6,3.95331,0.0513307
7,0.25,dense,1560,100.0,0.00115265,1.63373e-6,4.08919,0.0449839
8,0.25,dense,1560,1000.0,0.00115265,1.63373e-6,4.08919,0.0449839
9,0.25,dense,1560,10.0,0.00115265,1.63373e-6,4.08919,0.0449839


In [16]:
# Mean errors grouped by Hall layout
combine(groupby(df, :layout),
    :nprobes => first => :nprobes,
    :corr_pos_err_mm => mean => :mean_corr_pos_err_mm,
    :corr_pos_err_mm => maximum => :max_corr_pos_err_mm,
    :corr_tilt_err_deg => mean => :mean_corr_tilt_err_deg,
    :raw_pos_err_mm => mean => :mean_raw_pos_err_mm,
    :raw_tilt_err_deg => mean => :mean_raw_tilt_err_deg,
)

Row,layout,nprobes,mean_corr_pos_err_mm,max_corr_pos_err_mm,mean_corr_tilt_err_deg,mean_raw_pos_err_mm,mean_raw_tilt_err_deg
,String,Int64,Float64,Float64,Float64,Float64,Float64
1,coarse,180,0.00489032,0.00706429,1.79498e-5,4.21636,0.12336
2,medium,728,0.00122473,0.00136578,7.84478e-6,3.93767,0.120512
3,dense,1560,0.00125065,0.00140718,3.87176e-6,3.94586,0.116894


In [ ]:
# Mean errors grouped by current
combine(groupby(df, :current_A),
    :corr_pos_err_mm => mean => :mean_corr_pos_err_mm,
    :corr_tilt_err_deg => mean => :mean_corr_tilt_err_deg,
    :raw_pos_err_mm => mean => :mean_raw_pos_err_mm,
    :raw_tilt_err_deg => mean => :mean_raw_tilt_err_deg,
)

## Plots

In [ ]:
fig = Figure(size=(1100, 450))

ax1 = Axis(fig[1,1], xlabel="number of Hall probes", ylabel="position error [mm]", title="Bias-corrected position recovery")
for layout in unique(df.layout)
    sub = df[df.layout .== layout, :]
    scatter!(ax1, sub.nprobes, sub.corr_pos_err_mm; label=layout, markersize=12)
end
axislegend(ax1, position=:rt)

ax2 = Axis(fig[1,2], xlabel="transform scale", ylabel="tilt error [deg]", title="Bias-corrected tilt recovery")
for layout in unique(df.layout)
    sub = df[df.layout .== layout, :]
    scatter!(ax2, sub.scale, sub.corr_tilt_err_deg; label=layout, markersize=12)
end
axislegend(ax2, position=:rt)

fig

In [ ]:
fig2 = Figure(size=(1100, 450))

ax1 = Axis(fig2[1,1], xlabel="coil current [A]", ylabel="position error [mm]", title="Current sensitivity")
for scale in unique(df.scale)
    sub = df[df.scale .== scale, :]
    scatter!(ax1, sub.current_A, sub.corr_pos_err_mm; label="scale=$(scale)", markersize=12)
end
axislegend(ax1, position=:rt)

ax2 = Axis(fig2[1,2], xlabel="coil current [A]", ylabel="tilt error [deg]", title="Current sensitivity")
for scale in unique(df.scale)
    sub = df[df.scale .== scale, :]
    scatter!(ax2, sub.current_A, sub.corr_tilt_err_deg; label="scale=$(scale)", markersize=12)
end
axislegend(ax2, position=:rt)

fig2

## One-line interpretation helper

`corr_*` columns compare the bias-corrected magnetic-axis estimate to the actual geometric axis of the transformed field coil. These are the main recovery errors.

`raw_*` columns compare the uncorrected sinusoid/local magnetic-axis estimate to the actual transformed field-coil geometry.